In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from ragas.llms import LangchainLLMWrapper
from langchain_deepseek import ChatDeepSeek
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import AnswerAccuracy
from datasets import load_dataset
from copy import deepcopy
import numpy as np
from scipy.stats import ttest_ind
import torch
load_dotenv()

/home/yhuang/ondemand/paper_clean/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
df = pd.read_csv('../QA_datasets_classified_qa_eval/output_csv/HotpotQA_UND_gpt4o_Ragas.csv')
ragas_col = df['ragas_AA_short'].tolist()
pre_ragas = load_dataset("json", 
                        data_files="../QA_datasets_classified_qa_eval/intermediate/BASELINE_HotpotQA_UND_qa_gpt_with_squad_scores.jsonl",
                        split="all")
assert len(pre_ragas) == len(ragas_col), "Fatal Error: length mismatch"

In [3]:
with_ragas = pre_ragas.add_column('ragas_AA_short', ragas_col)
with_ragas.to_json("./intermediate/HotpotQA_UND_gpt4o_Ragas.jsonl", orient="records", lines=True)

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 66.71ba/s]


1709007

## Rewriting with Gemini
GPT-4o rewriting, then GPT-4o QA later

In [4]:
from helper_functions_qr import modification_in_batch

In [5]:
client = OpenAI(
    api_key=os.environ.get('GOOGLE_API_KEY'),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)
model = "gemini-2.5-flash"
input_file = "./intermediate/HotpotQA_UND_gpt4o_Ragas.jsonl"
output_file = "./intermediate/MODIFIED_HotpotQA_UND_gpt4o_Ragas.jsonl"



In [6]:
# 清空输出文件（如果存在）
if os.path.exists(output_file):
    os.remove(output_file)
    print(f"Cleared existing output file: {output_file}")

# 处理所有样本（按批次）
question_modification = modification_in_batch(input_file, output_file, 'answer', client, model)

Cleared existing output file: ./intermediate/MODIFIED_HotpotQA_UND_gpt4o_Ragas.jsonl
Total samples to process: 496
Batch size: 3


Processing batches:  32%|███▏      | 53/166 [1:02:03<1:28:47, 47.14s/it]

Error processing sample 160: Expecting ',' delimiter: line 3 column 122 (char 271)


Processing batches:  39%|███▊      | 64/166 [1:14:31<1:40:58, 59.40s/it] 

Error processing sample 194: Expecting ',' delimiter: line 3 column 228 (char 327)


...

Error processing sample 235: Invalid \escape: line 3 column 65 (char 187)


Processing batches:  52%|█████▏    | 86/166 [1:32:17<54:01, 40.52s/it]  

Error processing sample 259: Invalid \escape: line 3 column 121 (char 229)


Processing batches: 100%|██████████| 166/166 [2:34:53<00:00, 55.99s/it]   


All batch processing completed! Total processed: 496 samples
Results saved to: ./intermediate/MODIFIED_HotpotQA_UND_gpt4o_Ragas.jsonl


In [7]:
df_view = pd.DataFrame(question_modification)
df_view

,original_question,modified_question,short_answer,model_original_answer,classifier_reasoning,original_f1,original_em,original_AA
0,"In what year was the narrator of ""Blackadder's...","In what year was Rowan Atkinson, the actor who...",[1959],[1949],The query seeks the birth year of the narrator...,0.000000,0,0.00
1,What is the first two words of the fifth studi...,What are the first two words of the eighth stu...,[The Hungry],[The Patriotic],"The query references 'Joseph Edgar Foreman,' w...",0.000000,0,0.00
2,Robert Earl Holding owned an oil company that ...,Robert Earl Holding owned Sinclair Oil Corpora...,[Harry F. Sinclair],[David P. Smith],The query requires determining the original fo...,0.000000,0,0.00
3,In what county was Duffy Jackson born?,"In what county was Duffy Jackson, the jazz dru...",[Nassau County],[Suffolk County],The query seeks information about the birth co...,0.500000,0,0.00
4,When was the defending titlist of 2009–10 Biat...,"When was Emil Hegle Svendsen, the defending ti...",[27 January 1974],"[March 18, 1983]",The query seeks the birth date of the 'defendi...,0.000000,0,0.00
...,...,...,...,...,...,...,...,...
491,Where did Otto von Bismarck and Ludwig Friedri...,From which historical state or political entit...,[Prussia],"[- Otto von Bismarck: Schönhausen, Prussia (no...",The query asks for the places of origin of two...,0.117647,0,1.00
492,When was the singer of Miss Emily's Picture born?,When was the singer of the song 'Miss Emily's ...,"[August 11, 1946]","[John Conlee, the singer of 'Miss Emily's Pict...",The query requires identification of the singe...,0.375000,0,1.00
493,On what street was the hotel located where the...,On what street was the hotel located that expe...,[Peachtree Street],[South Virginia Street],The query requires identifying a specific fire...,0.400000,0,0.00
494,What is the original name of the place where T...,What is the original name of the place where T...,[Fort Saint Anthony],[Fort Snelling],The query seeks the 'original name' of the loc...,0.400000,0,0.25


## Modified queries QA using GPT-4o

### Loading modified data

In [8]:
modified_set = load_dataset("json",
    data_files="./intermediate/MODIFIED_HotpotQA_UND_gpt4o_Ragas.jsonl",
    split="all"  # 必须指定 split，否则默认返回 DatasetDict
)


#modified_set = modified_set.remove_columns(["model_new_answer"])

#modified_set.to_json(
    #"./intermediate/MODIFIED_HotpotQA_UND_gpt4o_Ragas.jsonl",
    #orient="records",
    #lines=True
#)

Generating train split: 496 examples [00:00, 12015.70 examples/s]


### Implementation

In [9]:
from helper_functions_qr import (ask_short_answer, run_batch_shortQA_api, batch_QA_with_progress)
client = OpenAI(
    api_key=os.environ.get('OPENAI_API_KEY')
)

In [10]:
modified_results = batch_QA_with_progress(
    modified_set,
    batch_fn=run_batch_shortQA_api,
    output_key="model_new_answer",
    fill_value=["error"],
    client=client,
    model="gpt-4o-2024-11-20",
    temperature=0.0
)

Running model_new_answer: 100%|██████████| 50/50 [08:04<00:00,  9.70s/it]


In [11]:
qa_modified = deepcopy(modified_set)
for key in modified_results:
    qa_modified = qa_modified.add_column(key, modified_results[key])

qa_modified.to_json("./intermediate/MODIFIED_HotpotQA_UND_gpt4o_Ragas.jsonl", orient="records", lines=True)
df_qa_modified = pd.read_json("./intermediate/MODIFIED_HotpotQA_UND_gpt4o_Ragas.jsonl", lines=True)
#df_qa_modified.to_csv('produced_files/modification_pilot_qa.csv')
df_qa_modified

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 170.20ba/s]


,original_question,modified_question,short_answer,model_original_answer,classifier_reasoning,original_f1,original_em,original_AA,model_new_answer
0,"In what year was the narrator of ""Blackadder's...","In what year was Rowan Atkinson, the actor who...",[1959],[1949],The query seeks the birth year of the narrator...,0.000000,0,0.00,[1955]
1,What is the first two words of the fifth studi...,What are the first two words of the eighth stu...,[The Hungry],[The Patriotic],"The query references 'Joseph Edgar Foreman,' w...",0.000000,0,0.00,[The Something]
2,Robert Earl Holding owned an oil company that ...,Robert Earl Holding owned Sinclair Oil Corpora...,[Harry F. Sinclair],[David P. Smith],The query requires determining the original fo...,0.000000,0,0.00,[Harry F. Sinclair]
3,In what county was Duffy Jackson born?,"In what county was Duffy Jackson, the jazz dru...",[Nassau County],[Suffolk County],The query seeks information about the birth co...,0.500000,0,0.00,[Shelby County]
4,When was the defending titlist of 2009–10 Biat...,"When was Emil Hegle Svendsen, the defending ti...",[27 January 1974],"[March 18, 1983]",The query seeks the birth date of the 'defendi...,0.000000,0,0.00,"[July 12, 1985]"
...,...,...,...,...,...,...,...,...,...
491,Where did Otto von Bismarck and Ludwig Friedri...,From which historical state or political entit...,[Prussia],"[- Otto von Bismarck: Schönhausen, Prussia (no...",The query asks for the places of origin of two...,0.117647,0,1.00,[Kingdom of Prussia]
492,When was the singer of Miss Emily's Picture born?,When was the singer of the song 'Miss Emily's ...,"[August 11, 1946]","[John Conlee, the singer of 'Miss Emily's Pict...",The query requires identification of the singe...,0.375000,0,1.00,"[Johnny Cash was born on February 26, 1932.]"
493,On what street was the hotel located where the...,On what street was the hotel located that expe...,[Peachtree Street],[South Virginia Street],The query requires identifying a specific fire...,0.400000,0,0.00,[North Lincoln Avenue]
494,What is the original name of the place where T...,What is the original name of the place where T...,[Fort Saint Anthony],[Fort Snelling],The query seeks the 'original name' of the loc...,0.400000,0,0.25,[Fort Snelling]


## Evaluations

### Squad EM+F1

In [12]:
# Helper functions updated, RESTART!!
from helper_functions_qr import evaluate_squad_per_sample_multi_ref_pred

modified_set = load_dataset("json",
    data_files="./intermediate/MODIFIED_HotpotQA_UND_gpt4o_Ragas.jsonl",
    split="all"  # 必须指定 split，否则默认返回 DatasetDict
)

qa_modified = deepcopy(modified_set)

Generating train split: 496 examples [00:00, 104049.95 examples/s]


In [13]:
squad_scored_modified, modified_f1_list, modified_em_list = evaluate_squad_per_sample_multi_ref_pred(qa_modified)
squad_scored_modified.to_json("./intermediate/MODIFIED_Gemini_HotpotQA_UND_gpt4o_new_squad.jsonl", orient="records", lines=True)

df = pd.read_json("./intermediate/MODIFIED_Gemini_HotpotQA_UND_gpt4o_new_squad.jsonl", lines=True)
df

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 210.37ba/s]


,original_question,modified_question,short_answer,model_original_answer,classifier_reasoning,original_f1,original_em,original_AA,model_new_answer,new_em,new_f1
0,"In what year was the narrator of ""Blackadder's...","In what year was Rowan Atkinson, the actor who...",[1959],[1949],The query seeks the birth year of the narrator...,0.000000,0,0.00,[1955],0,0.0
1,What is the first two words of the fifth studi...,What are the first two words of the eighth stu...,[The Hungry],[The Patriotic],"The query references 'Joseph Edgar Foreman,' w...",0.000000,0,0.00,[The Something],0,0.0
2,Robert Earl Holding owned an oil company that ...,Robert Earl Holding owned Sinclair Oil Corpora...,[Harry F. Sinclair],[David P. Smith],The query requires determining the original fo...,0.000000,0,0.00,[Harry F. Sinclair],1,1.0
3,In what county was Duffy Jackson born?,"In what county was Duffy Jackson, the jazz dru...",[Nassau County],[Suffolk County],The query seeks information about the birth co...,0.500000,0,0.00,[Shelby County],0,0.5
4,When was the defending titlist of 2009–10 Biat...,"When was Emil Hegle Svendsen, the defending ti...",[27 January 1974],"[March 18, 1983]",The query seeks the birth date of the 'defendi...,0.000000,0,0.00,"[July 12, 1985]",0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
491,Where did Otto von Bismarck and Ludwig Friedri...,From which historical state or political entit...,[Prussia],"[- Otto von Bismarck: Schönhausen, Prussia (no...",The query asks for the places of origin of two...,0.117647,0,1.00,[Kingdom of Prussia],0,0.5
492,When was the singer of Miss Emily's Picture born?,When was the singer of the song 'Miss Emily's ...,"[August 11, 1946]","[John Conlee, the singer of 'Miss Emily's Pict...",The query requires identification of the singe...,0.375000,0,1.00,"[Johnny Cash was born on February 26, 1932.]",0,0.0
493,On what street was the hotel located where the...,On what street was the hotel located that expe...,[Peachtree Street],[South Virginia Street],The query requires identifying a specific fire...,0.400000,0,0.00,[North Lincoln Avenue],0,0.0
494,What is the original name of the place where T...,What is the original name of the place where T...,[Fort Saint Anthony],[Fort Snelling],The query seeks the 'original name' of the loc...,0.400000,0,0.25,[Fort Snelling],0,0.4


In [14]:
modified_mean_em = np.mean(modified_em_list)  # em_scores: EM list per sample
modified_mean_f1 = np.mean(modified_f1_list)  # f1_scores F1 list per sample
print(f"New answers after modification Exact Match (avg): {modified_mean_em * 100:.2f}")
print(f"New answers after modification F1 Score (avg): {modified_mean_f1 * 100:.2f}")

original_em_list = qa_modified['original_em']
original_f1_list = qa_modified['original_f1']

original_mean_em = np.mean(original_em_list)  # em_scores: EM list per sample
original_mean_f1 = np.mean(original_f1_list)  # f1_scores F1 list per sample
print(f"Original answers Exact Match (avg): {original_mean_em * 100:.2f}")
print(f"Original answers F1 Score (avg): {original_mean_f1 * 100:.2f}")

f1_tstat, f1_pval = ttest_ind(modified_f1_list, original_f1_list, equal_var=False)
print(f"F1: t={f1_tstat:.3f}, p={f1_pval:.4f}")

em_tstat, em_pval = ttest_ind(modified_em_list, original_em_list, equal_var=False)
print(f"EM: t={em_tstat:.3f}, p={em_pval:.4f}")

New answers after modification Exact Match (avg): 34.48
New answers after modification F1 Score (avg): 51.82
Original answers Exact Match (avg): 21.98
Original answers F1 Score (avg): 34.64
F1: t=6.481, p=0.0000
EM: t=4.412, p=0.0000


### Ragas AA

In [15]:
from helper_functions_qr import answer_accuracy_modified
evaluator_llm = LangchainLLMWrapper(ChatDeepSeek(model="deepseek-chat", verbose=True, temperature=0))

In [16]:
squad_scored_modified = load_dataset("json",
    data_files="./intermediate/MODIFIED_Gemini_HotpotQA_UND_gpt4o_new_squad.jsonl",
    split="train")
result_with_AA = await answer_accuracy_modified(squad_scored_modified, evaluator_llm)
result_with_AA.to_csv("./output_csv/MODIFIED_Gemini_HotpotQA_UND_gpt4o_all_new_scores.csv")

Generating train split: 496 examples [00:00, 108082.65 examples/s]
Creating CSV from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 33.28ba/s]


478235

In [17]:
original_AA = list(result_with_AA["original_AA"])
modified_AA = list(result_with_AA["new_AA"])

original_mean_AA = np.mean(original_AA)
print(f"original AA (avg): {original_mean_AA * 100:.2f}")


modified_mean_AA = np.mean(modified_AA)
print(f"modified AA (avg): {modified_mean_AA * 100:.2f}")

AA_tstat, AA_pval = ttest_ind(modified_AA, original_AA, equal_var=False)
print(f"AA: t={AA_tstat:.3f}, p={AA_pval:.4f}")

original AA (avg): 42.19
modified AA (avg): 61.29
AA: t=6.589, p=0.0000


## Re-Classification

In [2]:
from helper_functions_qr import (
    batch_generate_responses_qwen3,
    get_judgments_from_responses,
    run_experiment,
    prepare_test_prompts)

print("HF_HOME:", os.environ.get("HF_HOME"))
print("HF_DATASETS_CACHE:", os.environ.get("HF_DATASETS_CACHE"))
print("HF_HUB_CACHE:", os.environ.get("HF_HUB_CACHE"))

HF_HOME: /scratch-local/yhuang/huggingface_cache
HF_DATASETS_CACHE: /scratch-local/yhuang/huggingface_cache/datasets
HF_HUB_CACHE: /scratch-local/yhuang/huggingface_cache/hub


### Loading data

In [3]:
reclassify_file = "./output_csv/MODIFIED_Gemini_HotpotQA_UND_gpt4o_all_new_scores.csv"
df_reclassify_file = pd.read_csv(reclassify_file)

### Loading model

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
Qwen3_4B = "Qwen/Qwen3-4B"
tokenizer = AutoTokenizer.from_pretrained(Qwen3_4B, padding_side='left')
model = AutoModelForCausalLM.from_pretrained(Qwen3_4B)
# 将模型移到可用设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = model.to(device)

Loading checkpoint shards: 100%|██████████| 3/3 [00:13<00:00,  4.50s/it]


cuda


### Prepare prompts

In [5]:
system_prompt = """
You are an expert analyst. Your task is to analyze and determine whether an input user query is "fully specified" or "underspecified".

"""

task_FS_UND = """
Analyze the following input user query:

{"query": "TARGET"}

Please provide your analysis in the following JSON format:

{"query": "TARGET", "reasoning": "[YOUR_DETAILED_REASONING]", "judgment": "[fully specified/underspecified]"}
"""

In [6]:
test_prompts = prepare_test_prompts(df_reclassify_file, task_FS_UND)
print(test_prompts[0])

Start preparing prompts...
# Testing data points: 496
Generation complete: 496 prompts
Average prompt length: 474 bytes (~118 tokens)

Analyze the following input user query:

{"query": "In what year was Rowan Atkinson, the actor who portrayed and narrated as Ebenezer Blackadder in "Blackadder's Christmas Carol," born?"}

Please provide your analysis in the following JSON format:

{"query": "In what year was Rowan Atkinson, the actor who portrayed and narrated as Ebenezer Blackadder in "Blackadder's Christmas Carol," born?", "reasoning": "[YOUR_DETAILED_REASONING]", "judgment": "[fully specified/underspecified]"}



In [7]:
test_df = run_experiment(tokenizer, model, test_prompts, system_prompt, df_reclassify_file)
test_df

100%|██████████| 100/100 [1:13:20<00:00, 44.00s/it]


,original_question,modified_question,short_answer,model_original_answer,classifier_reasoning,original_f1,original_em,original_AA,model_new_answer,new_em,new_f1,new_AA,MODIFIED_thinking,MODIFIED_model_response,MODIFIED_model_pred
0,"In what year was the narrator of ""Blackadder's...","In what year was Rowan Atkinson, the actor who...",['1959'],['1949'],The query seeks the birth year of the narrator...,0.000000,0,0.00,['1955'],0,0.0,0.0,"<think>\nOkay, let me try to figure out if thi...","{\n ""query"": ""In what year was Rowan Atkinson...",fully specified
1,What is the first two words of the fifth studi...,What are the first two words of the eighth stu...,['The Hungry'],['The Patriotic'],"The query references 'Joseph Edgar Foreman,' w...",0.000000,0,0.00,['The Something'],0,0.0,0.0,"<think>\nOkay, let's see. The user is asking f...","{\n ""query"": ""What are the first two words of...",fully specified
2,Robert Earl Holding owned an oil company that ...,Robert Earl Holding owned Sinclair Oil Corpora...,['Harry F. Sinclair'],['David P. Smith'],The query requires determining the original fo...,0.000000,0,0.00,['Harry F. Sinclair'],1,1.0,1.0,"<think>\nOkay, let's see. The user is asking a...","{\n ""query"": ""Robert Earl Holding owned Sincl...",fully specified
3,In what county was Duffy Jackson born?,"In what county was Duffy Jackson, the jazz dru...",['Nassau County'],['Suffolk County'],The query seeks information about the birth co...,0.500000,0,0.00,['Shelby County'],0,0.5,0.0,"<think>\nOkay, let's see. The user is asking a...","{\n ""query"": ""In what county was Duffy Jackso...",fully specified
4,When was the defending titlist of 2009–10 Biat...,"When was Emil Hegle Svendsen, the defending ti...",['27 January 1974'],"['March 18, 1983']",The query seeks the birth date of the 'defendi...,0.000000,0,0.00,"['July 12, 1985']",0,0.0,0.0,"<think>\nOkay, let's see. The user is asking f...","{\n ""query"": ""When was Emil Hegle Svendsen, t...",fully specified
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
491,Where did Otto von Bismarck and Ludwig Friedri...,From which historical state or political entit...,['Prussia'],"['- Otto von Bismarck: Schönhausen, Prussia (n...",The query asks for the places of origin of two...,0.117647,0,1.00,['Kingdom of Prussia'],0,0.5,1.0,"<think>\nOkay, let's see. The user is asking a...","{\n ""query"": ""From which historical state or ...",underspecified
492,When was the singer of Miss Emily's Picture born?,When was the singer of the song 'Miss Emily's ...,"['August 11, 1946']","[""John Conlee, the singer of 'Miss Emily's Pic...",The query requires identification of the singe...,0.375000,0,1.00,"['Johnny Cash was born on February 26, 1932.']",0,0.0,0.0,"<think>\nOkay, let me try to figure out if thi...","{\n ""query"": ""When was the singer of the song...",underspecified
493,On what street was the hotel located where the...,On what street was the hotel located that expe...,['Peachtree Street'],['South Virginia Street'],The query requires identifying a specific fire...,0.400000,0,0.00,['North Lincoln Avenue'],0,0.0,0.0,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""On what street was the hotel ...",fully specified
494,What is the original name of the place where T...,What is the original name of the place where T...,['Fort Saint Anthony'],['Fort Snelling'],The query seeks the 'original name' of the loc...,0.400000,0,0.25,['Fort Snelling'],0,0.4,0.0,"<think>\nOkay, let's tackle this query. The us...","{\n ""query"": ""What is the original name of th...",fully specified


In [8]:
test_df['MODIFIED_model_pred'].value_counts(normalize=True)


MODIFIED_model_pred
fully specified    0.693548
underspecified     0.306452
Name: proportion, dtype: float64

In [9]:
test_df['MODIFIED_model_pred'].value_counts()

MODIFIED_model_pred
fully specified    344
underspecified     152
Name: count, dtype: int64

In [10]:
test_df.to_csv('./output_csv/HotpotQA_UND_Gemini_rewritten_reclassified.csv')